# Controller stage schedules — linear against cubic (appendix artifacts)

<details>
<summary>What the two published populations were actually driven by, and what a parameter does at
each stage change.</summary>

The manuscript states the calibration runs "a staged schedule of increasing gains" and never says
what the schedules are. The two populations of Section 3.1 used different ones: under the linear
law the gain was raised once, from 0.1 to 0.25, under the cubic law it was escalated four times
across two orders of magnitude. This notebook emits the table and the figure that state it.

**Pure load — nothing is solved here.** It reads two already-saved populations
(`runConfig["schedules"][*]["path"]`) and the scenario JSON each was run under, so it costs two
file opens. The stage schedule is *not* stored in the HDF5: only the scenario NAME is
(`attrs["conf"]["scenario"]`), so the stage list has to be joined back from
`config/scenarios/`, and `scenarioRef` pins which revision of that file to read — a saved
population ran under whatever the JSON said at the time, and the file can have moved since.

Two traps are handled in the stage cell, both of which would silently misplace the marks:

- The saved clock is **rebased**. `runnerBatchSI` writes `globalT` over saved runs only, so a
  stage's `runsToIgnore` runs are integrated and discarded and the trace starts behind absolute
  simulated time. Marks are therefore built from `runsToSave` alone; the table reports the
  absolute cost, `(runsToIgnore + runsToSave) x runTime`.
- Both scenarios carry **zero-run stages**, which the `staged` path does not reject. Their gains
  are compiled into a model that is never stepped, so no parameter moves. A naive cumulative sum
  would draw a second line on top of their neighbour, so they are dropped from both artifacts.

The reconciliation print at the end of the stage cell is the gate: the resolved schedule's saved
sample count must equal the file's own `raw_coarse` length, or the wrong schedule was joined to
the wrong population and every mark is in the wrong place.

</details>

In [ ]:
# region -> runConfig (the ONE place run configuration lives; project single-config-surface rule)
####################################################################################################
# runConfig — the ONE place run configuration lives (project rule: see repo-root CLAUDE.md).
# Defined first so the device/precision block can be applied before JAX initialises below.
####################################################################################################
runConfig = {
    "plot": True,                 # False -> resolve + reconcile the schedules only, draw nothing

    # --- figure rows: which calibrated parameters to show ------------------------------------
    # Any name in the files' `param_names`. Three fit the manuscript column; the rows are drawn
    # in this order, and a name absent from a population leaves that panel empty.
    "params": ["R_As_Cs", "E_Hl", "C_Vs"],

    # --- figure columns: one saved population per control law --------------------------------
    # `scenarioRef` pins WHICH revision of the scenario JSON carries the schedule this file ran
    # under: None = the working tree, a git ref = that revision. Both laws read the working tree:
    # the linear schedule there is the four-stage one that produced the saved population
    # (off / 0.1 / 0.25 / off), which the committed revision no longer describes. The
    # reconciliation assert in the stage cell is what proves the join is right.
    "schedules": [
        {"label": "linear", "path": "notebookData/convergence/population_linear_batch.h5",
         "scenario": "sepsis_linear.json", "scenarioRef": None},
        {"label": "cubic",  "path": "notebookData/convergence/population_cubic_batch.h5",
         "scenario": "sepsis_cubic.json",  "scenarioRef": None},
    ],
    "trajDataset": "raw_coarse",  # "raw_coarse" = one sample per saved run, "raw" = every step

    # --- device / precision (applied in the Imports cell, before `import jax`) ----------------
    "device": {
        "useGpu":    False,       # GPU is off by default everywhere (project HARD RULE)
        "precision": "float64",
    },

    # --- analysis / plot ----------------------------------------------------------------------
    "analysis": {
        "divergenceLimit": 2000.0,   # |value| >= this in any observable/param = out of scope
        "fontSize":        7,        # final rendered size: the figure is drawn AT its target width
        "ylim":            "robust", # per-ROW framing pooled over the columns; None = autoscale
        "robustPct":       [1, 99],
        "showRuns":        None,     # cap on lane lines per panel (speed); None = every lane
        "stageLabels":     True,     # annotate each marked stage change with its multiplier
        # The x axis carries the absolute simulated clock, so the width of every stage on the
        # trace is its real cost and the two laws are read against the same seconds. The
        # alternative is "progress" + "log", which spreads the cubic law's early escalations at
        # the price of distorting those widths.
        "xAxis":           "seconds",  # "seconds" (absolute simulated clock) or "progress" ([0,1])
        "xScale":          "linear",   # "linear" or "log"; "log" requires xAxis "progress"
        "xLabel":          "simulated time (s)",
    },

    # --- paper artifacts (appendix table + figure) --------------------------------------------
    # `emit` ships False: flip it True for one run to write the artifacts, then revert it.
    "paper": {
        "emit":          True,
        "dir":           "EFC_Paper/revision/generated",
        "imageDir":      "EFC_Paper/revision/Images",
        "table":         "stageSchedule.tex",
        "figure":        "stageSchedule.png",
        "tableRef":      "tab:stageSchedule",
        "tableFontSize": "footnotesize",
        "colSpec":       "l r r r",
        "figSize":       [7.0, 3.6],  # inches, spans both columns: \includegraphics[width=\textwidth]
        "dpi":           200,
    },
}
# endregion

## Imports

<details>
<summary>Device/precision must be set from `runConfig` **before** `import jax`.</summary>

Nothing here solves the model, so JAX is only imported to keep the repo's import order intact for
the `library` modules that pull it in.

</details>

In [ ]:
# region -> imports + device/precision (must precede `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve
# ---- the relative notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
jax.config.update("jax_enable_x64", precision == "float64")

import numpy as np
import matplotlib.pyplot as plt
import h5py
import json
import subprocess

import library.run.runner as runner        # resolveStages (the same strategy dispatch the runs used)
import library.viz.plots as libPlots       # plotStageSchedule
import library.utils as utils              # loadScenario / configPath / generateLatexTableInline
from library.hdf5 import schema_pop        # good_run_mask (the one source for dropping bad lanes)

print(f"jax {jax.__version__} | stage-schedule comparison (pure load) | "
      f"x64={jax.config.jax_enable_x64}")
# endregion

## Stage schedules — resolve, drop the zero-run stages, reconcile against each population

<details>
<summary>The stage list is joined back from `config/scenarios/`, then checked against the file it is
supposed to describe.</summary>

`runner.resolveStages` is the same dispatch `runCalibration` and `runnerBatchSI` used, so the list
resolved here is the list that ran, `adaptive` strategies included.

Which multiplier reaches the live gain is decided by `controllerLaw`, not by the JSON key name:
with `multiplierL` absent, `calibratorUpdater` routes `multiplierC` into the LINEAR gain when
`controllerLaw == "linear"`. `liveMultiplier` mirrors that branch, so the table cannot drift from
the runner. The stage `description` strings are ignored on purpose: every calibrating stage in both
files reads "Aggressive calibration (multiplierC = 10.0)" whatever its actual multiplier is.

The reconciliation line must show `saved samples` matching each file's own `raw_coarse` length
(365 linear, 665 cubic). It raises rather than warns: a mismatch means this schedule did not
produce this population, and every stage mark drawn from it would be wrong.

</details>

In [ ]:
# region -> stage schedules: resolve per population, keep the stages that run, reconcile
def loadCalibration(entry):
    """(calibration conf, runTime, provenance string) for one schedule entry.

    `scenarioRef` pins WHICH revision of the scenario JSON to read. A saved population ran under
    whatever the file said at the time, and the file can have been edited since, so reading the
    working tree unconditionally would describe a schedule that never ran."""
    ref = entry.get("scenarioRef")
    if ref is None:
        conf = utils.loadScenario(entry["scenario"])
        src  = f"working tree {utils.configPath('scenarios', entry['scenario'])}"
    else:
        blob = subprocess.run(["git", "show", f"{ref}:config/scenarios/{entry['scenario']}"],
                              capture_output=True, text=True, check=True).stdout
        conf = json.loads(blob)
        src  = f"git {ref}:config/scenarios/{entry['scenario']}"
    return conf["calibration"], conf["shared"]["integration"]["runTime"], src


def liveMultiplier(stage, law):
    """The multiplier that actually reaches the live gain, mirroring `runner.calibratorUpdater`:
    with `multiplierL` absent, `controllerLaw` decides whether `multiplierC` feeds the cubic gain
    or the linear one, so the JSON key name is not the answer."""
    mC, mL = stage.get("multiplierC", 0.0), stage.get("multiplierL")
    if mL is None:
        return mC                                   # -> a_L when law is linear, a_C otherwise
    return mL if law == "linear" else mC


# The marks share the figure's x units, so they are built in whichever the axis carries.
secondsAxis = runConfig["analysis"].get("xAxis", "progress") == "seconds"

schedules = []
for entry in runConfig["schedules"]:
    cal, runTime, src = loadCalibration(entry)
    law    = cal.get("controllerLaw", "cubic")      # absent -> cubic (runner's own default)
    stages = runner.resolveStages(cal)

    # A stage integrating no runs is compiled but never stepped, so no parameter moves in it and
    # it has no width on the trace. Dropping it here keeps it out of BOTH artifacts and stops a
    # cumulative sum from drawing a duplicate mark on its neighbour.
    running = [s for s in stages
               if s.get("runsToIgnore", 0) + s.get("runsToSave", 0) > 0]
    dropped = len(stages) - len(running)

    savedSecs = np.array([s.get("runsToSave", 0) * runTime for s in running], dtype=float)
    totalSecs = float(savedSecs.sum())              # the SAVED clock, the one the trace carries
    absSecs   = [(s.get("runsToIgnore", 0) + s.get("runsToSave", 0)) * runTime for s in running]

    # marks at the START of every running stage but the first, in the axis's own units
    edges = np.cumsum(savedSecs)[:-1]
    marks = [(e if secondsAxis else e / totalSecs,
              ("off" if liveMultiplier(s, law) == 0 else f"{liveMultiplier(s, law):g}"))
             for e, s in zip(edges, running[1:]) if e > 0]

    with h5py.File(entry["path"], "r") as f:
        nSample  = f[runConfig["trajDataset"]].shape[1]
        dtDense  = float(json.loads(f.attrs["conf"]).get("dtDense", runTime))
        confName = json.loads(f.attrs["conf"]).get("scenario")
    expected = int(round(totalSecs / dtDense)) if runConfig["trajDataset"] == "raw" \
        else int(round(totalSecs / runTime))

    print(f"{entry['label']:7s} law={law:6s} | {len(stages)} stages, {len(running)} run "
          f"({dropped} zero-run dropped) | {int(sum(absSecs) / runTime)} runs, "
          f"{sum(absSecs):g} s simulated | saved samples {expected} vs file {nSample}")
    print(f"        schedule from {src} | file's own scenario name: {confName}")
    assert expected == nSample, (
        f"{entry['label']}: schedule gives {expected} saved samples, {entry['path']} holds "
        f"{nSample}. This schedule did not produce this population -- every stage mark would be "
        f"in the wrong place. Check `scenarioRef`.")

    schedules.append({**entry, "law": law, "runTime": runTime, "running": running,
                      "absSecs": absSecs, "savedSecs": savedSecs, "totalSecs": totalSecs,
                      "marks": marks, "nSample": nSample, "provenance": src})
    unit = "s" if secondsAxis else ""
    print("        marks at " + ", ".join(f"{p:{'.6g' if secondsAxis else '.4f'}}{unit} ({t})"
                                          for p, t in marks) + "\n")
# endregion

## Parameter traces — the per-lane trajectory blocks

<details>
<summary>One block per requested parameter per population, over the lanes that stayed in scope.</summary>

The calibrated parameters are columns of the saved state tensor, so a parameter's whole-calibration
trace across the population is `raw_coarse[:, :, raw_signal_names.index(name)]`. Lanes are filtered
by `schema_pop.good_run_mask` over the run-end observations and parameters, the project's one source
for dropping bad runs — a lane that left scope mid-run would set the y-range for every panel.

The prior box comes from each file's stored `problem` attribute, so the dotted sampling edges are
read from the run rather than restated here.

</details>

In [ ]:
# region -> trace load: per-lane parameter blocks + the prior box, per population
if runConfig["plot"]:
    divLimit = runConfig["analysis"]["divergenceLimit"]
    dset     = runConfig["trajDataset"]
    columns, ranges = [], {}

    for sch in schedules:
        with h5py.File(sch["path"], "r") as f:
            sig     = list(f["raw_signal_names"].asstr()[:])
            problem = json.loads(f.attrs["problem"])
            block   = np.asarray(f[dset][:, -1, :])           # (nLane, nSignal) run-end values
            obsIdx  = [sig.index(o) for o in f["observation_names"].asstr()[:] if o in sig]
            parIdx  = [sig.index(p) for p in f["param_names"].asstr()[:] if p in sig]
            good    = schema_pop.good_run_mask(block[:, obsIdx], divLimit,
                                               param_matrix=block[:, parIdx])
            traces = {}
            for name in runConfig["params"]:
                if name in sig:
                    traces[name] = np.asarray(f[dset][:, :, sig.index(name)])[good]
                else:
                    print(f"  NOTE {sch['label']}: {name} is not a saved signal, panel left empty")

        for name, bounds in zip(problem["names"], problem["bounds"]):
            ranges.setdefault(name, tuple(bounds))            # first population wins; the prior
                                                              # box is shared across the laws
        # x for this column. On the seconds axis each saved sample is the END of its run, so the
        # first sits one slice in and the last at the saved total — the same edges the marks use.
        nS = sch["nSample"]
        x  = (np.linspace(sch["totalSecs"] / nS, sch["totalSecs"], nS) if secondsAxis
              else np.linspace(0.0, 1.0, nS))
        columns.append({"label": sch["label"], "traces": traces, "stages": sch["marks"],
                        "prog": x})
        print(f"{sch['label']:7s} {int(good.sum())}/{len(good)} lanes in scope x "
              f"{nS} samples | {len(traces)} of {len(runConfig['params'])} params found "
              f"| x {x[0]:g} to {x[-1]:g}")
# endregion

## Figure — parameter trajectories with the stage changes marked

<details>
<summary>Rows are parameters, columns are control laws, dashed lines are the stage changes.</summary>

Drawn at the width it is included at (`paper.figSize`), so `analysis.fontSize` is the final
rendered size — a figure authored narrow and stretched across the text width renders 7 pt text at
the wrong size.

The x axis is the absolute simulated clock (`analysis.xAxis`), so each stage occupies its real
share of the run and the two laws are read against the same seconds: the linear schedule ends at
3650 s of saved time and the cubic at 6650 s. Each saved sample is the end of one 10 s forward
solve, so the first sample sits at 10 s and the marks fall on run boundaries.

The alternative is `xAxis` "progress" with `xScale` "log", which normalises each column to its own
length and spreads the cubic law's early escalations, at the cost of making the width of a stage
unreadable off the axis. The table below states the cost of every stage in runs either way.
Each row is framed on one pooled y-range, so the same parameter is on the same scale in both laws.

</details>

In [ ]:
# region -> figure: parameter trajectories, one row per parameter, one column per control law
if runConfig["plot"]:
    an = runConfig["analysis"]
    fig = libPlots.plotStageSchedule(
        columns, runConfig["params"], ranges=ranges,
        figSize=tuple(runConfig["paper"]["figSize"]), fontSize=an["fontSize"],
        ylim=an["ylim"], robustPct=tuple(an["robustPct"]), divLimit=an["divergenceLimit"],
        showRuns=an["showRuns"], stageLabels=an["stageLabels"], xScale=an["xScale"],
        xLabel=an.get("xLabel", "normalised progress"))
    plt.show()
# endregion

## Table — the two stage schedules

<details>
<summary>One row per stage that runs, the two laws separated by a rule.</summary>

Emitted through `utils.generateLatexTableInline`, the non-float `minipage` shape: a real
`\begin{table}` float is dropped without an error inside the manuscript's `multicols` body,
leaving a dangling `\ref` and a gap in the table numbering.

The gain column is the multiplier that reaches the live gain of whichever law is active, printed
as `off` where the controllers are inactive. Runs and simulated time are the absolute cost,
`(runsToIgnore + runsToSave) x runTime`, so the totals reproduce the step counts already published
in the four-method cost table.

</details>

In [ ]:
# region -> LaTeX stage-schedule table (both laws, one row per running stage)
if runConfig["plot"]:
    pc, rows, lawEnds = runConfig["paper"], {}, []
    for sch in schedules:
        for k, (stage, secs) in enumerate(zip(sch["running"], sch["absSecs"]), start=1):
            m = liveMultiplier(stage, sch["law"])
            rows[f"{sch['label'].capitalize()} {k}"] = [
                "off" if m == 0 else f"{m:g}",
                f"{int(secs / sch['runTime'])}",
                f"{secs:g}",
            ]
        lawEnds.append(f"{sch['label'].capitalize()} {len(sch['running'])}")

    # a rule after every law block but the last, which already has the \bottomrule
    totals = ", ".join(f"{s['label']} {int(sum(s['absSecs']) / s['runTime'])}" for s in schedules)
    stageTable = utils.generateLatexTableInline(
        rows, ["", "Gain multiplier", "Runs", "Simulated time (s)"],
        ref=pc["tableRef"], colSpec=pc["colSpec"], fontSize=pc["tableFontSize"],
        ruleRows=lawEnds[:-1],
        # Caption kept SHORT: the appendix prose already carries the comparison and the totals,
        # and the whole subsection is held to a 200-word budget.
        caption=(
            "Calibration stages of the two control laws, in order. The gain multiplier scales the "
            "gain of the active law, $a_L$ for the linear law and $a_C$ for the cubic; "
            "\\emph{off} is a settling stage. Each run is one "
            f"{schedules[0]['runTime']:g}\\,s forward solve, so the run column is the cost of the "
            "stage. Stages that integrate no runs are omitted."))
    print(stageTable)
# endregion

## Paper artifacts — write the table and the figure

<details>
<summary>Guarded by `paper.emit`, which ships False.</summary>

Flip `paper.emit` True for one run to write both artifacts, then revert it, so a re-run of the
notebook cannot silently rewrite a manuscript figure.

</details>

In [ ]:
# region -> emit the appendix table + figure into the paper tree (guarded by paper.emit)
if runConfig["plot"]:
    pc = runConfig["paper"]
    if pc["emit"]:
        os.makedirs(pc["dir"], exist_ok=True)
        os.makedirs(pc["imageDir"], exist_ok=True)
        outTable = os.path.join(pc["dir"], pc["table"])
        with open(outTable, "w") as fh:
            fh.write(stageTable)
        print(f"wrote {outTable}")
        outFig = os.path.join(pc["imageDir"], pc["figure"])
        fig.savefig(outFig, dpi=pc["dpi"], bbox_inches="tight")
        print(f"wrote {outFig}")
    else:
        print("paper.emit is False -- table/figure not written")
# endregion